## Imports

In [ ]:
# Data handling libraries
import json
import numpy as np
import pandas as pd
from pandas import json_normalize

# Natural Language Processing (NLP) libraries
from nltk.corpus import stopwords

# Scikit-learn modeling libraries
from sklearn.dummy import DummyClassifier # For baseline model
from sklearn.feature_extraction.text import TfidfVectorizer # To convert text to numbers
from sklearn.linear_model import LogisticRegression # The classifier model
from sklearn.metrics import accuracy_score, classification_report # For evaluation
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score # For splitting and validating
from sklearn.pipeline import Pipeline # To chain processing steps

## Loading data

In [ ]:
# Load the training data from a JSON Lines file (one JSON object per line)
train_data = pd.read_json('../../data/train.jsonl', lines=True)
# The tweet data is nested. json_normalize flattens the nested JSON into columns.
train_data = json_normalize(train_data.to_dict(orient='records'))

# Load the Kaggle test data (which we will make predictions on)
kaggle_data = pd.read_json('../../data/kaggle_test.jsonl', lines=True)
# Also normalize the Kaggle data
kaggle_data = json_normalize(kaggle_data.to_dict(orient='records'))


# Separate features from the target variable for the training set
X_train = train_data.drop('label', axis=1)
y_train = train_data['label']

X_kaggle = kaggle_data

## Extract

In [ ]:
# Define a function to get the full text from a tweet object.
# Tweets can be truncated, storing the full version in 'extended_tweet.full_text'.
def extract_full_text(tweet):
    # Start with the standard 'text' field
    text = tweet['text']
    # Check if the 'extended_tweet.full_text' field exists (is not NaN)
    if not pd.isna(tweet['extended_tweet.full_text']):
        # If it exists, it's the full text, so use it instead
        text = tweet['extended_tweet.full_text']
    return text

# Apply this function to every row (axis=1) in the training data
X_train['full_text'] = X_train.apply(lambda tweet: extract_full_text(tweet), axis=1)
# Apply the same function to the Kaggle test data
X_kaggle['full_text'] = X_kaggle.apply(lambda tweet: extract_full_text(tweet), axis=1)

## Main part

In [ ]:
"""
ULTIMATE KAGGLE STACKING SOLUTION (CORRIGÉ)
Method: 5-Fold Cross-Validation with Out-of-Fold (OOF) Stacking.
Level 1: RoBERTa, DistilBERT, XGBoost, LightGBM, CatBoost.
Level 2: Neural Network Meta-Learner trained on OOF predictions.
"""

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
# CORRECTION 1: Suppression d'AdamW ici
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
# CORRECTION 1: Import correct d'AdamW
from torch.optim import AdamW 

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm
import warnings
import gc
import os

warnings.filterwarnings('ignore')

# Configuration
N_FOLDS = 3  # Standard for Kaggle
SEED = 42
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')

def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)
print(f"🔥 Hardware selected: {DEVICE}")

# ============================================================================
# FEATURE ENGINEERING (Optimized)
# ============================================================================

def safe_get(d, key, default=0):
    if isinstance(d, dict):
        return d.get(key, default)
    return default

def get_full_text(row):
    text = row.get('text', '') or row.get('full_text', '')
    if not text and isinstance(row.get('extended_tweet'), dict):
        text = row.get('extended_tweet', {}).get('full_text', '')
    return str(text) if text else ''

def get_user_description(row):
    user = row.get('user', {})
    return str(user.get('description', '')) if isinstance(user, dict) else ''

def extract_features(df):
    """Extract tabular features for GBM models"""
    print("   Extracting tabular features...")
    features = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc="   Processing"):
        user = row.get('user', {})
        followers = safe_get(user, 'followers_count', 0)
        friends = max(safe_get(user, 'friends_count', 1), 1)
        statuses = max(safe_get(user, 'statuses_count', 1), 1)
        listed = safe_get(user, 'listed_count', 0)
        
        try:
            created_at = safe_get(user, 'created_at', '')
            year = int(created_at.split()[-1]) if created_at else 2024
            account_age = max(2025 - year, 1)
        except:
            account_age = 1
        
        text = get_full_text(row)
        entities = row.get('entities', {})
        
        feat = {
            'followers_log': np.log1p(followers),
            'friends_log': np.log1p(friends),
            'ratio_fol_friend': followers / friends,
            'listed_log': np.log1p(listed),
            'statuses_log': np.log1p(statuses),
            'account_age': account_age,
            'tweets_per_year': statuses / account_age,
            'verified': int(safe_get(user, 'verified', False)),
            'default_profile': int(safe_get(user, 'default_profile', True)),
            'default_profile_image': int(safe_get(user, 'default_profile_image', True)),
            'retweet_count_log': np.log1p(row.get('retweet_count', 0) or 0),
            'favorite_count_log': np.log1p(row.get('favorite_count', 0) or 0),
            'text_len': len(text),
            'word_count': len(text.split()),
            'hashtag_count': len(entities.get('hashtags', []) if isinstance(entities, dict) else []),
            'url_count': len(entities.get('urls', []) if isinstance(entities, dict) else []),
            'has_description': int(bool(safe_get(user, 'description', ''))),
            'desc_len': len(str(safe_get(user, 'description', '')))
        }
        features.append(feat)
    
    return pd.DataFrame(features).fillna(0).replace([np.inf, -np.inf], 0)

# ============================================================================
# DEEP LEARNING UTILS
# ============================================================================

class TweetDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        inputs = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        # squeeze the batch dimension returned by tokenizer when return_tensors='pt'
        item = {
            'input_ids': inputs['input_ids'].squeeze(0),
            'attention_mask': inputs['attention_mask'].squeeze(0)
        }
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

def train_transformer_fold(model_name, X_train, y_train, X_val, y_val, X_test):
    """Train a transformer for one fold and return val + test preds"""
    
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    
    # CORRECTION 2: Ajout de ignore_mismatched_sizes=True
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, 
        num_labels=2,
        ignore_mismatched_sizes=True 
    )
    model.to(DEVICE)
    
    train_dataset = TweetDataset(X_train, y_train, tokenizer)
    val_dataset = TweetDataset(X_val, y_val, tokenizer)
    test_dataset = TweetDataset(X_test, None, tokenizer)
    
    train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=32)
    test_loader = DataLoader(test_dataset, batch_size=32)
    
    optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.05)
    epochs = 2 # Keep low for stacking speed, rely on ensemble
    
    for epoch in range(epochs):
        model.train()
        for batch in tqdm(train_loader, desc=f"      DL Train Ep {epoch+1}", leave=False):
            input_ids = batch['input_ids'].to(DEVICE)
            mask = batch['attention_mask'].to(DEVICE)
            labels = batch['labels'].to(DEVICE)
            
            outputs = model(input_ids, attention_mask=mask, labels=labels)
            loss = outputs.loss
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            
    # Inference
    model.eval()
    
    def predict(loader):
        probs = []
        with torch.no_grad():
            for batch in loader:
                input_ids = batch['input_ids'].to(DEVICE)
                mask = batch['attention_mask'].to(DEVICE)
                out = model(input_ids, attention_mask=mask)
                probs.extend(torch.softmax(out.logits, dim=1)[:, 1].cpu().numpy())
        return np.array(probs)
    
    val_preds = predict(val_loader)
    test_preds = predict(test_loader)
    
    del model, optimizer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()
    
    return val_preds, test_preds

# ============================================================================
# META LEARNER
# ============================================================================

class MetaNetwork(nn.Module):
    def __init__(self, input_dim):
        super(MetaNetwork, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 2)
        )

    def forward(self, x):
        return self.net(x)

def train_meta_learner(X_train, y_train, X_test):
    # Normalize features
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)
    
    X_train_t = torch.FloatTensor(X_train).to(DEVICE)
    y_train_t = torch.LongTensor(y_train).to(DEVICE)
    X_test_t = torch.FloatTensor(X_test).to(DEVICE)
    
    model = MetaNetwork(X_train.shape[1]).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    
    # Train simple NN
    model.train()
    for epoch in range(50):
        optimizer.zero_grad()
        out = model(X_train_t)
        loss = criterion(out, y_train_t)
        loss.backward()
        optimizer.step()
        
    model.eval()
    with torch.no_grad():
        test_logits = model(X_test_t)
        test_probs = torch.softmax(test_logits, dim=1)[:, 1].cpu().numpy()
        
    return test_probs

# ============================================================================
# MAIN PIPELINE
# ============================================================================

print("🚀 LOADING DATA...")
# Assure-toi que ces fichiers sont bien dans le dossier courant
train_raw = pd.read_json('../../data/train.jsonl', lines=True)
test_raw = pd.read_json('../../data/kaggle_test.jsonl', lines=True)

# Prepare Inputs
print("📝 PREPARING INPUTS...")
# Text for DL
train_texts = [f"{get_full_text(r)} [SEP] {get_user_description(r)}" for _, r in train_raw.iterrows()]
test_texts = [f"{get_full_text(r)} [SEP] {get_user_description(r)}" for _, r in test_raw.iterrows()]

# Features for GBM
train_feat = extract_features(train_raw)
test_feat = extract_features(test_raw)

y = train_raw['label'].values

# K-FOLD INITIALIZATION
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

# Store Out-of-Fold Predictions (Train size x Num Models)
# Models: 0:RoBERTa, 1:DistilBERT, 2:LGBM, 3:XGB, 4:CatBoost
oof_preds = np.zeros((len(train_raw), 5)) 

# Store Test Predictions (Test size x Num Models) - Accumulator
test_preds_accum = np.zeros((len(test_raw), 5))

print(f"\n⚡ STARTING {N_FOLDS}-FOLD STACKING...")

for fold, (train_idx, val_idx) in enumerate(skf.split(train_texts, y)):
    print(f"\n{'='*20} FOLD {fold+1}/{N_FOLDS} {'='*20}")
    
    # Split Data
    X_tr_txt = [train_texts[i] for i in train_idx]
    X_val_txt = [train_texts[i] for i in val_idx]
    
    X_tr_feat = train_feat.iloc[train_idx]
    X_val_feat = train_feat.iloc[val_idx]
    
    y_tr, y_val = y[train_idx], y[val_idx]
    
    # --- MODEL 1: RoBERTa ---
    print("   1. RoBERTa (Fine-tuning)...")
    val_p, test_p = train_transformer_fold('cardiffnlp/twitter-roberta-base-sentiment-latest', 
                                            X_tr_txt, y_tr, X_val_txt, y_val, test_texts)
    oof_preds[val_idx, 0] = val_p
    test_preds_accum[:, 0] += test_p / N_FOLDS
    
    # --- MODEL 2: DistilBERT ---
    print("   2. DistilBERT (Fine-tuning)...")
    val_p, test_p = train_transformer_fold('distilbert-base-uncased', 
                                            X_tr_txt, y_tr, X_val_txt, y_val, test_texts)
    oof_preds[val_idx, 1] = val_p
    test_preds_accum[:, 1] += test_p / N_FOLDS
    
    # --- MODEL 3: LightGBM ---
    print("   3. LightGBM...")
    lgb_model = lgb.LGBMClassifier(n_estimators=1000, learning_rate=0.02, verbose=-1)
    lgb_model.fit(X_tr_feat, y_tr, eval_set=[(X_val_feat, y_val)], callbacks=[lgb.early_stopping(50, verbose=False)])
    oof_preds[val_idx, 2] = lgb_model.predict_proba(X_val_feat)[:, 1]
    test_preds_accum[:, 2] += lgb_model.predict_proba(test_feat)[:, 1] / N_FOLDS
    
    # --- MODEL 4: XGBoost ---
    print("   4. XGBoost...")
    xgb_model = xgb.XGBClassifier(n_estimators=1000, learning_rate=0.02, n_jobs=-1, eval_metric='logloss')
    xgb_model.fit(X_tr_feat, y_tr, eval_set=[(X_val_feat, y_val)], verbose=False)
    oof_preds[val_idx, 3] = xgb_model.predict_proba(X_val_feat)[:, 1]
    test_preds_accum[:, 3] += xgb_model.predict_proba(test_feat)[:, 1] / N_FOLDS

    # --- MODEL 5: CatBoost ---
    print("   5. CatBoost...")
    cat_model = CatBoostClassifier(iterations=1000, learning_rate=0.02, verbose=0, allow_writing_files=False)
    cat_model.fit(X_tr_feat, y_tr, eval_set=(X_val_feat, y_val))
    oof_preds[val_idx, 4] = cat_model.predict_proba(X_val_feat)[:, 1]
    test_preds_accum[:, 4] += cat_model.predict_proba(test_feat)[:, 1] / N_FOLDS

    # Fold Score
    # Simple average of OOF preds for this fold just to check progress
    avg_fold_pred = oof_preds[val_idx].mean(axis=1)
    acc = accuracy_score(y_val, (avg_fold_pred > 0.5).astype(int))
    print(f"   ✓ Fold {fold+1} Blended Accuracy: {acc:.4f}")


    # Submission partielle
    print("Submission partielle pour le fold en cours...")
    X_meta_train_partiel = np.hstack([oof_preds, train_feat.values])
    X_meta_test_partiel = np.hstack([test_preds_accum, test_feat.values])
    partial_probs = train_meta_learner(X_meta_train_partiel, y, X_meta_test_partiel)
    partial_preds = (partial_probs > 0.5).astype(int)

    submission_partielle = pd.DataFrame({
        'ID': test_raw['challenge_id'] if 'challenge_id' in test_raw.columns else test_raw["id_str"],
        'Prediction': partial_preds
    })
    
    partial_filename = f"submission_oof_stacking_partial_fold_{fold+1}.csv"
    submission_partielle.to_csv(partial_filename, index=False)
    print(f"✅ DONE! Saved to {partial_filename}")
    print("="*50)


# ========================================================================
# LEVEL 2: META LEARNER (THE "META FRIEND" PART)
# ========================================================================
print("\n🧠 TRAINING META-LEARNER (Level 2)...")

# Input for meta learner: OOF Predictions + Original Features
X_meta_train = np.hstack([oof_preds, train_feat.values])
X_meta_test = np.hstack([test_preds_accum, test_feat.values])

# Train Neural Network on complete OOF data
final_probs = train_meta_learner(X_meta_train, y, X_meta_test)

final_preds = (final_probs > 0.5).astype(int)

global FINAL_PROBS 
FINAL_PROBS = final_probs  # For potential debugging
global FINAL_PREDS 
FINAL_PREDS = final_preds  # For potential debugging

# ========================================================================
# SUBMISSION
# ========================================================================
submission = pd.DataFrame({
    'ID': test_raw['challenge_id'] if 'challenge_id' in test_raw.columns else test_raw["id_str"],
    'Prediction': final_preds
})

filename = 'submission_oof_stacking_final.csv'
submission.to_csv(filename, index=False)

print("\n" + "="*50)
print(f"✅ DONE! Saved to {filename}")
print(f"Mean Prediction Probability: {final_probs.mean():.4f}")
print(f"Predicted Class 1 Ratio: {final_preds.mean():.4f}")
print("="*50)

In [ ]:
# SUBMISSION
# ========================================================================

test_raw = pd.read_json('../../data/kaggle_test.jsonl', lines=True)


submission = pd.DataFrame({
    'id': test_raw['id_str'],
    'target': FINAL_PREDS
})

filename = 'submission_oof_stacking.csv'
submission.to_csv(filename, index=False)

print("\n" + "="*50)
print(f"✅ DONE! Saved to {filename}")
print(f"Mean Prediction Probability: {FINAL_PROBS.mean():.4f}")
print(f"Predicted Class 1 Ratio: {FINAL_PREDS.mean():.4f}")
print("="*50)